In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Knowledge Base
documents = [
    {
        "id": "HR-001",
        "text": "Employees receive 26 annual leave days."
    },
    {
        "id": "HR-002",
        "text": "Leave requests must be approved by the manager."
    },
    {
        "id": "HR-003",
        "text": "Employees have supplementary health insurance."
    },
    {
        "id": "HR-004",
        "text": "Monthly salaries are paid at the end of each month."
    },
    {
        "id": "HR-005",
        "text": "Employees can work remotely two days per week."
    },
    {
        "id": "HR-006",
        "text": "Employees can request unpaid leave."
    }
]

# 2. Load Embedding Model
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 3. Generate Document Embeddings
document_texts = [doc["text"] for doc in documents]

document_embeddings = model.encode(
    document_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# 4. Retrieve Relevant Chunks
def retrieve_chunks(query, top_k=3):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    ranked_indices = np.argsort(scores)[::-1]

    results = []

    for index in ranked_indices[:top_k]:
        results.append({
            "id": documents[index]["id"],
            "text": documents[index]["text"],
            "score": float(scores[index])
        })

    return results


# 5. Construct Context
def construct_context(retrieved_chunks):

    context_parts = []

    for chunk in retrieved_chunks:

        context_parts.append(
            f"[Source: {chunk['id']}]\n"
            f"{chunk['text']}"
        )

    return "\n\n".join(context_parts)


# 6. Execute RAG Retrieval + Context Construction
query = "How many annual leave days do employees receive?"

retrieved_chunks = retrieve_chunks(
    query=query,
    top_k=3
)

context = construct_context(retrieved_chunks)

# 7. Display Results
print("USER QUERY:")
print(query)

print("\nRETRIEVED CHUNKS:")
for chunk in retrieved_chunks:
    print(f"{chunk['id']} | Score: {chunk['score']:.4f}")
    print(chunk["text"])

print("\nCONSTRUCTED CONTEXT:")
print(context)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 861.75it/s]


USER QUERY:
How many annual leave days do employees receive?

RETRIEVED CHUNKS:
HR-001 | Score: 0.7969
Employees receive 26 annual leave days.
HR-006 | Score: 0.6642
Employees can request unpaid leave.
HR-005 | Score: 0.5376
Employees can work remotely two days per week.

CONSTRUCTED CONTEXT:
[Source: HR-001]
Employees receive 26 annual leave days.

[Source: HR-006]
Employees can request unpaid leave.

[Source: HR-005]
Employees can work remotely two days per week.
